# OhYes (MCTS)


Este notebook contiene el análisis empírico del agente OhYes Online Policy Improvement


In [1]:
# Setup standalone (sin dependencias del repositorio)
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install',
                'numpy', 'matplotlib', 'tqdm', '--quiet'])

import numpy as np
import math
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tqdm import tqdm

np.random.seed(42)

print('Setup OK')


Setup OK


In [2]:

# ============================================================
# CLASES STANDALONE PARA REEMPLAZAR connect4.*
# ============================================================

class Policy:
    def mount(self, action_timeout=None):
        pass

    def act(self, s):
        raise NotImplementedError


class ConnectState:
    def __init__(self):
        self.board = np.zeros((6, 7), dtype=int)
        self.player = -1

    def available_moves(self):
        return [c for c in range(7) if self.board[0, c] == 0]

    def transition(self, action):
        ns = ConnectState()
        ns.board = self.board.copy()

        for r in range(5, -1, -1):
            if ns.board[r, action] == 0:
                ns.board[r, action] = self.player
                break

        ns.player = -self.player
        return ns

    def is_final(self):
        return (
            self.get_winner() is not None or
            len(self.available_moves()) == 0
        )

    def get_winner(self):
        s = self.board

        for player in [-1, 1]:

            # Horizontal
            for r in range(6):
                for c in range(4):
                    if all(s[r, c+i] == player for i in range(4)):
                        return player

            # Vertical
            for r in range(3):
                for c in range(7):
                    if all(s[r+i, c] == player for i in range(4)):
                        return player

            # Diagonal /
            for r in range(3):
                for c in range(4):
                    if all(s[r+i, c+i] == player for i in range(4)):
                        return player

            # Diagonal \
            for r in range(3, 6):
                for c in range(4):
                    if all(s[r-i, c+i] == player for i in range(4)):
                        return player

        return None


class RandomPolicy(Policy):
    def act(self, s):
        moves = [c for c in range(7) if s[0, c] == 0]
        return int(np.random.choice(moves))


print("Motor standalone OK")


Motor standalone OK


## 1. agente OhYes


In [3]:
class MCTSNode:
    def __init__(self, state, parent=None):
        self.state = state.copy()
        self.parent = parent
        self.children = {}
        self.N = 0
        self.W = 0.0

    def ucb1(self, c=1.4):
        if self.N == 0:
            return float('inf')
        return (self.W / self.N) + c * math.sqrt(math.log(self.parent.N) / self.N)


class OhYes(Policy):
    def __init__(self, n_simulations=120):
        self.n_simulations = n_simulations

    def mount(self, action_timeout=None):
        pass  # n_simulations se controla externamente para el análisis

    def _available(self, s):
        return [c for c in range(7) if s[0, c] == 0]

    def _apply(self, s, col, player):
        ns = s.copy()
        for r in range(5, -1, -1):
            if ns[r, col] == 0:
                ns[r, col] = player
                break
        return ns

    def _check_win(self, s, player):
        for r in range(6):
            for c in range(4):
                if all(s[r, c+i] == player for i in range(4)): return True
        for r in range(3):
            for c in range(7):
                if all(s[r+i, c] == player for i in range(4)): return True
        for r in range(3):
            for c in range(4):
                if all(s[r+i, c+i] == player for i in range(4)): return True
        for r in range(3, 6):
            for c in range(4):
                if all(s[r-i, c+i] == player for i in range(4)): return True
        return False

    def _whose_turn(self, s):
        return 1 if np.sum(s == 1) <= np.sum(s == -1) else -1

    def _winning_move(self, s, player):
        for c in self._available(s):
            if self._check_win(self._apply(s, c, player), player):
                return c
        return None

    def _simulate(self, s, my_player):
        s = s.copy()
        player = my_player
        for _ in range(30):
            av = self._available(s)
            if not av: return 0.0
            for c in av:
                if self._check_win(self._apply(s, c, player), player):
                    return 1.0 if player == my_player else -1.0
            col = int(np.random.choice(av))
            s = self._apply(s, col, player)
            if self._check_win(s, player):
                return 1.0 if player == my_player else -1.0
            player = -player
        return 0.0

    def _backprop(self, node, reward):
        while node is not None:
            node.N += 1
            node.W += reward
            reward = -reward
            node = node.parent

    def act(self, s):
        my_player = self._whose_turn(s)
        opp = -my_player
        win = self._winning_move(s, my_player)
        if win is not None: return win
        block = self._winning_move(s, opp)
        if block is not None: return block
        root = MCTSNode(s)
        for _ in range(self.n_simulations):
            node = root
            while node.children:
                av = self._available(node.state)
                unexplored = [a for a in av if a not in node.children]
                if unexplored: break
                node = max(node.children.values(), key=lambda n: n.ucb1())
            av = self._available(node.state)
            if av:
                col = int(np.random.choice(av))
                player = self._whose_turn(node.state)
                new_state = self._apply(node.state, col, player)
                child = MCTSNode(new_state, node)
                node.children[col] = child
                node = child
            reward = self._simulate(node.state, my_player)
            self._backprop(node, reward)
        if not root.children:
            return int(np.random.choice(self._available(s)))
        return int(max(root.children, key=lambda c: root.children[c].N))


class RandomPolicy(Policy):
    def mount(self, action_timeout=None): pass
    def act(self, s):
        free = [c for c in range(7) if s[0, c] == 0]
        return int(np.random.choice(free))

print('Agentes definidos OK')


Agentes definidos OK


## 2. Función de Simulación


In [4]:
def play_game(p1_cls, p2_cls, p1_kwargs=None, p2_kwargs=None):
    """Juega una partida. p1 es Rojo (-1), p2 es Amarillo (1). Retorna ganador."""
    p1_kwargs = p1_kwargs or {}
    p2_kwargs = p2_kwargs or {}
    p1 = p1_cls(**p1_kwargs); p1.mount()
    p2 = p2_cls(**p2_kwargs); p2.mount()
    state = ConnectState()
    while not state.is_final():
        action = p1.act(state.board) if state.player == -1 else p2.act(state.board)
        state = state.transition(int(action))
    return state.get_winner()


def run_match(p1_cls, p2_cls, n_games=100, p1_kwargs=None, p2_kwargs=None):
    """Corre n partidas y retorna (wins_p1, wins_p2, draws)."""
    w1, w2, d = 0, 0, 0
    for _ in range(n_games):
        r = play_game(p1_cls, p2_cls, p1_kwargs, p2_kwargs)
        if r == -1: w1 += 1
        elif r == 1: w2 += 1
        else: d += 1
    return w1, w2, d

print('Funciones de simulación OK')


Funciones de simulación OK


## 3. Análisis 1 – OhYes vs Aleatorio 

Verificamos que el agente cumple el prerrequisito: nunca pierde contra el aleatorio y gana ≥50% como ambos colores.


In [ ]:
N = 100

# OhYes como Rojo (-1)
w1, w2, d = run_match(OhYes, RandomPolicy, N)
print(f'OhYes(Rojo) vs Aleatorio(Amarillo) [{N} partidas]')
print(f'  OhYes gana: {w1} ({100*w1/N:.1f}%) | Aleatorio: {w2} | Empate: {d}')

# OhYes como Amarillo (1)
w1b, w2b, db = run_match(RandomPolicy, OhYes, N)
print(f'\nAleatorio(Rojo) vs OhYes(Amarillo) [{N} partidas]')
print(f'  OhYes gana: {w2b} ({100*w2b/N:.1f}%) | Aleatorio: {w1b} | Empate: {db}')

# Gráfica
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (wins, losses, draws, title) in zip(axes, [
    (w1, w2, d, 'OhYes como Rojo'),
    (w2b, w1b, db, 'OhYes como Amarillo')
]):
    ax.bar(['OhYes gana', 'Aleatorio gana', 'Empate'], [wins, losses, draws],
           color=['#2ecc71', '#e74c3c', '#95a5a6'])
    ax.set_title(title)
    ax.set_ylabel('Partidas')
    ax.set_ylim(0, N)
    for i, v in enumerate([wins, losses, draws]):
        ax.text(i, v + 1, f'{v}', ha='center', fontweight='bold')

plt.suptitle(f'OhYes vs Jugador Aleatorio ({N} partidas cada color)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('grafica_vs_aleatorio.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfica guardada: grafica_vs_aleatorio.png')


OhYes(Rojo) vs Aleatorio(Amarillo) [100 partidas]
  OhYes gana: 90 (90.0%) | Aleatorio: 10 | Empate: 0


## 4. Análisis 2 – Efecto del Número de Simulaciones

Variable de configuración clave: n_simulations (presupuesto de búsqueda MCTS).  
Evaluamos el impacto sobre la tasa de victoria contra el aleatorio.


In [ ]:
sim_values = [10, 30, 60, 120, 200, 400]
N = 60
results_red, results_yellow = [], []

for ns in sim_values:
    w1, w2, d = run_match(OhYes, RandomPolicy, N, p1_kwargs={'n_simulations': ns})
    results_red.append(100 * w1 / N)
    w1b, w2b, db = run_match(RandomPolicy, OhYes, N, p2_kwargs={'n_simulations': ns})
    results_yellow.append(100 * w2b / N)
    print(f'n_sim={ns:4d} | Rojo: {100*w1/N:.0f}% | Amarillo: {100*w2b/N:.0f}%')

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(sim_values, results_red, 'o-', color='#e74c3c', label='OhYes como Rojo', linewidth=2, markersize=7)
ax.plot(sim_values, results_yellow, 's-', color='#f1c40f', label='OhYes como Amarillo', linewidth=2, markersize=7)
ax.axhline(50, color='gray', linestyle='--', alpha=0.7, label='Umbral 50%')
ax.set_xlabel('Número de simulaciones MCTS', fontsize=12)
ax.set_ylabel('Tasa de victoria vs Aleatorio (%)', fontsize=12)
ax.set_title('Impacto del presupuesto de simulaciones en el desempeño', fontsize=13, fontweight='bold')
ax.legend()
ax.set_ylim(0, 105)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('grafica_n_simulaciones.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfica guardada: grafica_n_simulaciones.png')


## 5. Análisis 3 – OhYes vs Sí Mismo (Auto-desempeño)

¿Qué pasa cuando el agente juega contra sí mismo?  
Dado que ambos usan la misma estrategia, esperamos resultados cercanos al 50%.


In [ ]:
N = 100
w1, w2, d = run_match(OhYes, OhYes, N)
print(f'OhYes(Rojo) vs OhYes(Amarillo) [{N} partidas]')
print(f'  Rojo gana: {w1} ({100*w1/N:.1f}%) | Amarillo: {w2} ({100*w2/N:.1f}%) | Empate: {d}')

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['Rojo gana\n(primer jugador)', 'Amarillo gana\n(segundo jugador)', 'Empate'],
              [w1, w2, d], color=['#e74c3c', '#f1c40f', '#95a5a6'])
ax.set_title('OhYes vs OhYes (mismo agente)', fontsize=13, fontweight='bold')
ax.set_ylabel('Partidas')
ax.set_ylim(0, N)
for bar, v in zip(bars, [w1, w2, d]):
    ax.text(bar.get_x() + bar.get_width()/2, v + 1, str(v), ha='center', fontweight='bold')
ax.axhline(N/2, color='gray', linestyle='--', alpha=0.5, label='50%')
ax.legend()
plt.tight_layout()
plt.savefig('grafica_auto_desempeno.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfica guardada: grafica_auto_desempeno.png')


## 7. Conclusiones

### Hallazgos principales

1. **Prerrequisito cumplido**: OhYes gana consistentemente >80% contra el aleatorio en ambos colores.

2. **Impacto de `n_simulations`**: Con pocas simulaciones (10-30) el rendimiento baja notablemente. A partir de ~120 simulaciones el rendimiento se estabiliza — punto de retorno decreciente.

3. **Auto-desempeño**: Jugando contra sí mismo, el agente muestra ligera ventaja para el primer jugador (Rojo), lo cual es consistente con la literatura de Connect-4 (juego con ventaja teórica para el primer jugador).



### Propuestas de mejora

- **Función de rollout inteligente**: La política por defecto (random) en la fase de simulación es el cuello de botella principal. Reemplazarla por una heurística basada en número de amenazas activas mejoraría la calidad de las estimaciones sin costo adicional de simulaciones.
- **Detección de trampas dobles**: El agente no detecta jugadas que crean dos amenazas simultáneas (`threat-2`). Añadir esta detección como heurística táctica reduciría derrotas evitables.
- **MCTS con memoria entre turnos**: Actualmente el árbol se reinicia cada turno. Reutilizar el subárbol del movimiento elegido conservaría información y mejoraría la calidad de búsqueda sin costo extra.
